In [ ]:
import json, os, glob, tqdm
from PIL import Image
import numpy as np
import subprocess

def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

# FFHQ: HDRI_sota_sj.json

In [ ]:
ax = 1
method_json = f"/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/RotateSH_figures/ffhq_rotateSH_userstudy_axis{ax}.json"
samples = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/HDRI_sota_sj.json"

method = [f"hou21_rotate_axis={ax}", f"hou22_rotate_axis={ax}", f"iclight_rotate_512x512_map_centered_axis{ax}", f"ours_difareli_rotate_rot{ax}", f"ours_difareli++_oneshot_rotate_rot{ax}_tomax"]
os.makedirs("./vids/", exist_ok=True)
os.makedirs(f'./vids/all_outputs/res/', exist_ok=True)
os.makedirs(f'./vids/all_outputs/misc/', exist_ok=True)

with open(method_json, 'r') as f:
    method_json = json.load(f)

with open(samples, 'r') as f:
    samples = json.load(f)['pair']

no_f1 = True

for sample_id, sample in tqdm.tqdm(samples.items()):
    src = sample['src']
    dst = sample['dst']
    for m in method:
        img_dir = method_json[m]['img_dir']
        n_frames = method_json[m]['n_frame']
        os.makedirs(f'./vids/axis={ax}/{src}_{dst}/{m}/', exist_ok=True)
        if "hou21" in m or "hou22" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        elif "iclight" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        elif "difareli++" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
            shadow = sort_by_frame(glob.glob(f'{img_path}/dst_shadm_shad_f*.png'))[1:]
            render = sort_by_frame(glob.glob(f'{img_path}/dst_ren_f*.png'))[1:]
            for _, (s, r) in enumerate(zip(shadow, render)):
                i = int(s.split('/')[-1].split('frame')[-1].split('.')[0])
                assert i == int(r.split('/')[-1].split('frame')[-1].split('.')[0])
                os.system(f'cp {s} ./vids/axis={ax}/{src}_{dst}/{m}/dst_shadm_shad_frame_{i:04d}.png')
                os.system(f'cp {r} ./vids/axis={ax}/{src}_{dst}/{m}/dst_ren_frame_{i:04d}.png')

            assert len(relit) == len(shadow) == len(render)
            for j in range(len(relit)):
                r_tmp = np.array(Image.open(relit[j]))
                h, w, _ = r_tmp.shape
                s_tmp = np.array(Image.open(shadow[j]).resize((h//2, w//2)))
                ren_tmp = np.array(Image.open(render[j]).resize((h//2, w//2)))
                
                out_comb = np.concatenate((r_tmp, np.concatenate((ren_tmp, s_tmp), axis=0)), axis=1)
                Image.fromarray(out_comb).save(f'./vids/axis={ax}/{src}_{dst}/{m}/res+cond_frame_{j:04d}.png')
                
            cmds = [f'ffmpeg -y -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}/dst_shadm_shad_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids/axis={ax}/{src}_{dst}/{m}/shadow.mp4',
                    f'ffmpeg -y -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}/dst_ren_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids/axis={ax}/{src}_{dst}/{m}/render.mp4',
                    # Vertical stack them (render on top of shadow)
                    f'ffmpeg -y -i ./vids/axis={ax}/{src}_{dst}/{m}/render.mp4 -i ./vids/axis={ax}/{src}_{dst}/{m}/shadow.mp4 -filter_complex vstack ./vids/axis={ax}/{src}_{dst}/{m}/cond.mp4',
                    f'ffmpeg -y -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}/res+cond_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids/axis={ax}/{src}_{dst}/{m}/res+cond.mp4'
            ]
            for c in cmds:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError as e:
                    print("An error occurred:", e)
        
        
            
        elif "difareli" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        
        if len(relit) == 0:
            # Create 0 images for mockup and match the number of frames
            for i in range(1, int(n_frames)):
                img = Image.new('RGB', (256, 256), color='black')
                img.save(f'./vids/axis={ax}/{src}_{dst}/{m}/res_frame_{i:04d}.png')
            continue
        
        os.makedirs(f'./vids/axis={ax}/{src}_{dst}/{m}/', exist_ok=True)
        # Copy all images to a new folder
        for img in relit:
            i = int(img.split('/')[-1].split('frame')[-1].split('.')[0])
            os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/{m}/res_frame_{i:04d}.png')
        
        #NOTE: Generate video high compression with 24 fps
        cmd = f'ffmpeg -y -r 24 -i ./vids/axis={ax}/{src}_{dst}/{m}/res_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids/axis={ax}/{src}_{dst}/res_{m}.mp4'
        try:
            subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except subprocess.CalledProcessError as e:
            print("An error occurred:", e)


    # Copy the ball image
    ball = sorted(glob.glob(f"/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis={ax}/valid/{sample_id}_src={src}_dst={dst}/n_step={n_frames}/ball/m_*.png"))
    assert len(ball) == int(n_frames)
    if no_f1:
        ball = ball[1:]
    os.makedirs(f'./vids/axis={ax}/{src}_{dst}/ball/', exist_ok=True)
    for img in ball:
        os.system(f'cp {img} ./vids/axis={ax}/{src}_{dst}/ball/{img.split("/")[-1]}')

    cmd = f'ffmpeg -y -r 24 -i ./vids/axis={ax}/{src}_{dst}/ball/m_%03d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./vids/axis={ax}/{src}_{dst}/ball.mp4'
    try:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print("An error occurred:", e)

    # Concatenate all videos vertically following the method order
    video_list = []
    for m in method:
        if os.path.exists(f'./vids/axis={ax}/{src}_{dst}/res_{m}.mp4'):
            if "difareli++" in m:
                video_list.append(f'./vids/axis={ax}/{src}_{dst}/{m}/res+cond.mp4')
            else:
                video_list.append(f'./vids/axis={ax}/{src}_{dst}/res_{m}.mp4')
    # video_list.append(f'./vids/axis={ax}/{src}_{dst}/ball.mp4')
    
    num_videos = len(video_list)
    # Generate FFmpeg inputs and filter_complex dynamically
    input_files = " ".join(f"-i {path}" for path in video_list)
    filter_complex = "".join(f"[{i}:v:0]" for i in range(num_videos)) + f"hstack=inputs={num_videos}"

    # Output file
    output_file = f"./vids/axis={ax}/{src}_{dst}/res_all.mp4"

    # Construct the FFmpeg command
    command = f"ffmpeg {input_files} -filter_complex \"{filter_complex}\" -c:v libx264 -crf 17 -preset slow {output_file}"

    # Execute the command
    try:
        subprocess.run(command, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print("An error occurred:", e)

    # Create the reverse video of res_all.mp4, ball.mp4 and cond.mp4
    os.system(f'ffmpeg -i {output_file} -vf reverse ./vids/axis={ax}/{src}_{dst}/res_all_reverse.mp4')
    os.system(f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}/cond.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}/cond_reverse.mp4')
    os.system(f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/ball.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/ball_reverse.mp4')
    os.system(f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}/res+cond.mp4 -vf reverse ./vids/axis={ax}/{src}_{dst}/{m}/res+cond_reverse.mp4')

    # Time concatenation
    filter_cat = f'-filter_complex "[0:v][1:v]concat=n=2:v=1:a=0"'
    os.system(f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/res_all.mp4 -i ./vids/axis={ax}/{src}_{dst}/res_all_reverse.mp4 {filter_cat} ./vids/axis={ax}/{src}_{dst}/res_all_concat.mp4')
    os.system(f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/ball.mp4 -i ./vids/axis={ax}/{src}_{dst}/ball_reverse.mp4 {filter_cat} ./vids/axis={ax}/{src}_{dst}/ball_concat.mp4')
    os.system(f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}/cond.mp4 -i ./vids/axis={ax}/{src}_{dst}/{m}/cond_reverse.mp4 {filter_cat} ./vids/axis={ax}/{src}_{dst}/{m}/cond_concat.mp4')
    os.system(f'ffmpeg -i ./vids/axis={ax}/{src}_{dst}/{m}/res+cond.mp4 -i ./vids/axis={ax}/{src}_{dst}/{m}/res+cond_reverse.mp4 {filter_cat} ./vids/axis={ax}/{src}_{dst}/{m}/res+cond_concat.mp4')

    # Copy the res_all.mp4 and ball.mp4 to ./vids/all_outputs/ and rename them with {src}_{dst}_{ax}.mp4
    os.system(f'cp /data/mint/DPM_Dataset/ffhq_256_with_anno/ffhq_256/valid/{src} ./vids/all_outputs/res/{src}')
    os.system(f'cp ./vids/axis={ax}/{src}_{dst}/res_all_concat.mp4 ./vids/all_outputs/res/{src}_{dst}_{ax}.mp4')
    os.system(f'cp ./vids/axis={ax}/{src}_{dst}/ball_concat.mp4 ./vids/all_outputs/res/{src}_{dst}_{ax}_ball.mp4')
    os.system(f'cp ./vids/axis={ax}/{src}_{dst}/{m}/cond_concat.mp4 ./vids/all_outputs/misc/{src}_{dst}_{ax}_cond.mp4')
    os.system(f'cp ./vids/axis={ax}/{src}_{dst}/{m}/res+cond_concat.mp4 ./vids/all_outputs/misc/{src}_{dst}_{ax}_res+cond.mp4')


  0%|          | 0/20 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

# FFHQ: selected_rotate_SH.json

In [14]:
ax = 2
method_json = f"/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/RotateSH_figures/ffhq_rotateSH_with_baseline_axis2.json"
samples = "/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/RotateSH_figures/sample_json_with_frameid/selected_rotate_RT.json"

# method = [f"hou21_rotate_axis={ax}", f"hou22_rotate_axis={ax}", f"iclight_rotate_512x512_map_centered_axis{ax}", f"ours_256_difareli_rotate_rot{ax}", f"ours_difareli++_oneshot_rotate_rot{ax}_tomax"]
method = [f"hou21_rotate_axis={ax}", f"hou22_rotate_axis={ax}", f"iclight_rotate_512x512_map_centered_axis{ax}", f"ours_256_difareli_rotate_rot{ax}", f"ours_256_difareli++_rotate_rot{ax}_tomax"]
root_dir = "./vids_selected_rotate_SH"
os.makedirs(root_dir, exist_ok=True)
os.makedirs(f'./{root_dir}/all_outputs/res/', exist_ok=True)
os.makedirs(f'./{root_dir}/all_outputs/misc/', exist_ok=True)

with open(method_json, 'r') as f:
    method_json = json.load(f)

with open(samples, 'r') as f:
    samples = json.load(f)['pair']

no_f1 = True

for sample_id, sample in tqdm.tqdm(samples.items()):
    src = sample['src']
    dst = sample['dst']
    for m in method:
        img_dir = method_json[m]['img_dir']
        n_frames = method_json[m]['n_frame']
        os.makedirs(f'./{root_dir}/axis={ax}/{src}_{dst}/{m}/', exist_ok=True)
        if "hou21" in m or "hou22" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        elif "iclight" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        elif "difareli++" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
            # shadow = sort_by_frame(glob.glob(f'{img_path}/dst_shadm_shad_f*.png'))[1:]
            # render = sort_by_frame(glob.glob(f'{img_path}/dst_ren_f*.png'))[1:]
            
            shadow = sort_by_frame(glob.glob(f'{img_path}/shadm_shad_f*.png'))[1:]
            render = sort_by_frame(glob.glob(f'{img_path}/ren_f*.png'))[1:]
            for _, (s, r) in enumerate(zip(shadow, render)):
                i = int(s.split('/')[-1].split('frame')[-1].split('.')[0])
                assert i == int(r.split('/')[-1].split('frame')[-1].split('.')[0])
                os.system(f'cp {s} ./{root_dir}/axis={ax}/{src}_{dst}/{m}/dst_shadm_shad_frame_{i:04d}.png')
                os.system(f'cp {r} ./{root_dir}/axis={ax}/{src}_{dst}/{m}/dst_ren_frame_{i:04d}.png')

            assert len(relit) == len(shadow) == len(render)
            for j in range(len(relit)):
                r_tmp = np.array(Image.open(relit[j]))
                h, w, _ = r_tmp.shape
                s_tmp = np.array(Image.open(shadow[j]).resize((h//2, w//2)))
                ren_tmp = np.array(Image.open(render[j]).resize((h//2, w//2)))
                
                out_comb = np.concatenate((r_tmp, np.concatenate((ren_tmp, s_tmp), axis=0)), axis=1)
                Image.fromarray(out_comb).save(f'./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond_frame_{j:04d}.png')
                
            cmds = [f'ffmpeg -y -r 24 -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/dst_shadm_shad_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./{root_dir}/axis={ax}/{src}_{dst}/{m}/shadow.mp4',
                    f'ffmpeg -y -r 24 -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/dst_ren_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./{root_dir}/axis={ax}/{src}_{dst}/{m}/render.mp4',
                    # Vertical stack them (render on top of shadow)
                    f'ffmpeg -y -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/render.mp4 -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/shadow.mp4 -filter_complex vstack ./{root_dir}/axis={ax}/{src}_{dst}/{m}/cond.mp4',
                    f'ffmpeg -y -r 24 -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond.mp4'
            ]
            for c in cmds:
                try:
                    subprocess.run(c, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                except subprocess.CalledProcessError as e:
                    print("An error occurred:", e)
        
        
            
        elif "difareli" in m:
            img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
            relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        
        if len(relit) == 0:
            # Create 0 images for mockup and match the number of frames
            for i in range(1, int(n_frames)):
                img = Image.new('RGB', (256, 256), color='black')
                img.save(f'./{root_dir}/axis={ax}/{src}_{dst}/{m}/res_frame_{i:04d}.png')
            continue
        
        os.makedirs(f'./{root_dir}/axis={ax}/{src}_{dst}/{m}/', exist_ok=True)
        # Copy all images to a new folder
        for img in relit:
            i = int(img.split('/')[-1].split('frame')[-1].split('.')[0])
            os.system(f'cp {img} ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res_frame_{i:04d}.png')
        
        #NOTE: Generate video high compression with 24 fps
        cmd = f'ffmpeg -y -r 24 -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res_frame_%04d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./{root_dir}/axis={ax}/{src}_{dst}/res_{m}.mp4'
        try:
            subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except subprocess.CalledProcessError as e:
            print("An error occurred:", e)


    # Copy the ball image
    ball = sorted(glob.glob(f"/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_selected_rotateSH/axis={ax}/valid/{sample_id}_src={src}_dst={dst}/n_step={n_frames}/ball/m_*.png"))
    assert len(ball) == int(n_frames)
    if no_f1:
        ball = ball[1:]
    os.makedirs(f'./{root_dir}/axis={ax}/{src}_{dst}/ball/', exist_ok=True)
    for img in ball:
        os.system(f'cp {img} ./{root_dir}/axis={ax}/{src}_{dst}/ball/{img.split("/")[-1]}')

    cmd = f'ffmpeg -y -r 24 -i ./{root_dir}/axis={ax}/{src}_{dst}/ball/m_%03d.png -c:v libx264 -preset slow -crf 17 -vf fps=24 -pix_fmt yuv420p ./{root_dir}/axis={ax}/{src}_{dst}/ball.mp4'
    try:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print("An error occurred:", e)

    # Concatenate all videos vertically following the method order
    video_list = []
    for m in method:
        if os.path.exists(f'./{root_dir}/axis={ax}/{src}_{dst}/res_{m}.mp4'):
            if "difareli++" in m:
                video_list.append(f'./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond.mp4')
            else:
                video_list.append(f'./{root_dir}/axis={ax}/{src}_{dst}/res_{m}.mp4')
    
    num_videos = len(video_list)
    # Generate FFmpeg inputs and filter_complex dynamically
    input_files = " ".join(f"-i {path}" for path in video_list)
    filter_complex = "".join(f"[{i}:v:0]" for i in range(num_videos)) + f"hstack=inputs={num_videos}"

    # Output file
    output_file = f"./{root_dir}/axis={ax}/{src}_{dst}/res_all.mp4"

    # Construct the FFmpeg command
    command = f"ffmpeg {input_files} -filter_complex \"{filter_complex}\" -c:v libx264 -crf 17 -preset slow {output_file}"

    # Execute the command
    try:
        subprocess.run(command, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print("An error occurred:", e)

    # Create the reverse video of res_all.mp4, ball.mp4 and cond.mp4
    os.system(f'ffmpeg -i {output_file} -vf reverse ./{root_dir}/axis={ax}/{src}_{dst}/res_all_reverse.mp4')
    os.system(f'ffmpeg -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/cond.mp4 -vf reverse ./{root_dir}/axis={ax}/{src}_{dst}/{m}/cond_reverse.mp4')
    os.system(f'ffmpeg -i ./{root_dir}/axis={ax}/{src}_{dst}/ball.mp4 -vf reverse ./{root_dir}/axis={ax}/{src}_{dst}/ball_reverse.mp4')
    os.system(f'ffmpeg -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond.mp4 -vf reverse ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond_reverse.mp4')

    # Time concatenation
    filter_cat = f'-filter_complex "[0:v][1:v]concat=n=2:v=1:a=0"'
    os.system(f'ffmpeg -i ./{root_dir}/axis={ax}/{src}_{dst}/res_all.mp4 -i ./{root_dir}/axis={ax}/{src}_{dst}/res_all_reverse.mp4 {filter_cat} ./{root_dir}/axis={ax}/{src}_{dst}/res_all_concat.mp4')
    os.system(f'ffmpeg -i ./{root_dir}/axis={ax}/{src}_{dst}/ball.mp4 -i ./{root_dir}/axis={ax}/{src}_{dst}/ball_reverse.mp4 {filter_cat} ./{root_dir}/axis={ax}/{src}_{dst}/ball_concat.mp4')
    os.system(f'ffmpeg -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/cond.mp4 -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/cond_reverse.mp4 {filter_cat} ./{root_dir}/axis={ax}/{src}_{dst}/{m}/cond_concat.mp4')
    os.system(f'ffmpeg -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond.mp4 -i ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond_reverse.mp4 {filter_cat} ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond_concat.mp4')

    # Copy the res_all.mp4 and ball.mp4 to ./{root_dir}/all_outputs/ and rename them with {src}_{dst}_{ax}.mp4
    os.system(f'cp /data/mint/DPM_Dataset/ffhq_256_with_anno/ffhq_256/valid/{src} ./{root_dir}/all_outputs/res/{src}')
    os.system(f'cp ./{root_dir}/axis={ax}/{src}_{dst}/res_all_concat.mp4 ./{root_dir}/all_outputs/res/{src}_{dst}_{ax}.mp4')
    os.system(f'cp ./{root_dir}/axis={ax}/{src}_{dst}/ball_concat.mp4 ./{root_dir}/all_outputs/res/{src}_{dst}_{ax}_ball.mp4')
    os.system(f'cp ./{root_dir}/axis={ax}/{src}_{dst}/{m}/cond_concat.mp4 ./{root_dir}/all_outputs/misc/{src}_{dst}_{ax}_cond.mp4')
    os.system(f'cp ./{root_dir}/axis={ax}/{src}_{dst}/{m}/res+cond_concat.mp4 ./{root_dir}/all_outputs/misc/{src}_{dst}_{ax}_res+cond.mp4')
    assert False


  0%|          | 0/16 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

AssertionError: 